In [0]:
create or replace table healthcare.gold.doctor_claim_summary as 
select 
  d.doctor_id,
  d.employee_code,
  d.full_name,
  d.department,
  count(v.visit_id) as total_visits,
  count(c.claim_id) as total_claims,
  sum(c.claim_amount) as total_claim_value,
  round(avg(c.claim_amount),2) as avg_claim_value,
  round(sum(case when lower(trim(c.status)) = 'approved' then 1 else 0 end)* 100.0 / 
  nullif(count(distinct c.claim_id), 0),2) AS approval_rate_pct
from healthcare.silver.doctors d
left join healthcare.silver.visits v 
  on d.doctor_id = v.doctor_id 
left join healthcare.silver.claims c
  on v.visit_id = c.visit_id 
group by 
  d.doctor_id, d.employee_code, d.full_name, d.department;

num_affected_rows,num_inserted_rows


In [0]:
select * from healthcare.gold.doctor_claim_summary;

doctor_id,employee_code,full_name,department,total_visits,total_claims,total_claim_value,avg_claim_value,approval_rate_pct
26,EMP-026,Chanchal Swaminathan,Radiology,38,25,2705821.09,108232.84,32.00
36,EMP-036,Bina Sidhu,OPD,56,26,2969544.18,114213.24,23.08
46,EMP-046,Turvi Gola,Other,51,21,1828935.19,87092.15,28.57
25,EMP-025,Bhavani Dube,ICU,50,17,2353178.55,138422.27,29.41
31,EMP-031,Gautam Sami,Emergency,39,16,1758692.65,109918.29,37.50
16,EMP-016,Sudiksha Luthra,Other,42,13,1334824.89,102678.84,7.69
7,EMP-007,Girik Jani,Radiology,52,22,2265717.92,102987.18,27.27
6,EMP-006,Manbir Dasgupta,Emergency,34,13,1622997.78,124845.98,30.77
32,EMP-032,Quincy Saraf,Radiology,51,17,1152027.06,67766.30,35.29
3,EMP-003,Jack Talwar,ICU,36,11,1344515.37,122228.67,0.00


In [0]:
create or replace table healthcare.gold.patient_cost_profile as
select
p.patient_id,
p.mrn,
p.full_name,
count(distinct v.visit_id) as total_visits,
count(t.treatment_id) as total_treatments,
sum(t.cost) as total_treatment_cost,
round(sum(t.cost) / nullif(count(distinct v.visit_id),0),2) as avg_cost_per_visit,
dense_rank() over(
  order by sum(t.cost) desc 
) as cost_rank
from healthcare.silver.treatments t 
join healthcare.silver.visits v 
on t.visit_id = v.visit_id 
join healthcare.silver.patients p 
on v.patient_id = p.patient_id 
group by 
p.patient_id,p.mrn,p.full_name;
select * from healthcare.gold.patient_cost_profile;

patient_id,mrn,full_name,total_visits,total_treatments,total_treatment_cost,avg_cost_per_visit,cost_rank
35,MRN-00035,Sanaya Dutt,10,22,544495,54449.50,1
393,MRN-00393,Nandini Sachdev,7,17,490252,70036.00,2
467,MRN-00467,Megha Rattan,5,18,485383,97076.60,3
391,MRN-00391,Ekta Setty,6,15,478999,79833.17,4
155,MRN-00155,Maanas Sheth,7,17,464902,66414.57,5
233,MRN-00233,Gunbir Chada,10,18,464322,46432.20,6
190,MRN-00190,Aradhana Date,7,16,461729,65961.29,7
244,MRN-00244,Abhiram Kapur,8,16,455089,56886.13,8
236,MRN-00236,Mekhala Nanda,7,16,447761,63965.86,9
351,MRN-00351,Lavanya Bhagat,5,18,437344,87468.80,10


In [0]:
create or replace table healthcare.gold.daily_visit_summary as
select 
v.visit_date,
count(distinct v.visit_id) as total_visits,
coalesce(sum(t.cost),0) as total_treatment_cost,
round(avg(count(distinct v.visit_id)) over (
  order by v.visit_date
  rows between 6 preceding and current row
),2) as rolling_7d_avg_visits
from healthcare.silver.visits v 
left join healthcare.silver.treatments t 
on v.visit_id = t.visit_id 
group by v.visit_date;
select * from healthcare.gold.daily_visit_summary;

visit_date,total_visits,total_treatment_cost,rolling_7d_avg_visits
2024-04-05,2,95933,2.0
2024-04-06,3,119574,2.5
2024-04-07,1,0,2.0
2024-04-08,2,168891,2.0
2024-04-09,3,121920,2.2
2024-04-10,4,139595,2.5
2024-04-11,5,369484,2.86
2024-04-12,5,249137,3.29
2024-04-13,4,157669,3.43
2024-04-14,3,170546,3.71


In [0]:
create or replace table healthcare.gold.insurer_perfomance as 
select 
insurer_name,
count(distinct claim_id) as total_claims,
count(case when status = 'Approved' then 1 end) as approved_claims,
count(case when status = 'Under Review' then 1 end) as under_review_claims,
count(case when status = 'Pending' then 1 end) as pending_claims,
count(case when status = 'Submitted' then 1 end) as submitted_claims,
count(case when status = 'Rejected' then 1 end) as rejected_claims,
round(avg(datediff(settled_date,claim_date)),2) as avg_settlement_days
from healthcare.silver.claims 
group by insurer_name;
select * from healthcare.gold.insurer_perfomance;

insurer_name,total_claims,approved_claims,under_review_claims,pending_claims,submitted_claims,rejected_claims,avg_settlement_days
New India Assurance,106,26,13,20,23,24,39.81
Aditya Birla Health,77,13,10,17,16,21,44.13
Bajaj Allianz Health,90,18,13,25,13,21,47.57
Other,99,20,23,19,14,23,42.09
Tata Aig Health,99,25,18,17,17,22,46.0
Care Health Insurance,86,16,14,16,21,19,42.45
Icici Lombard Health,97,23,16,25,19,14,43.3
United India Insurance,82,15,19,14,17,17,48.92
Star Health Insurance,84,16,16,13,21,18,47.87
Hdfc Ergo Health,90,12,23,16,16,23,44.83
